In [1]:
import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

from djimaging.user.amacrine_cells_neuromodulation.notebooks.helper_save_function import save_with_meta
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [3]:
current_date = datetime.date.today().strftime("%Y%m%d")
print(current_date)

20261013


In [4]:
df = pd.read_pickle(f"../datasets/Processed/Averages_ClusterLabels_20261013.pkl")
df.shape

(5888, 48)

In [11]:
# Determine the set of cond2 values per (date, field) group
group_conditions = df.groupby(["date", "field"])["cond2"].apply(lambda s: set(s.unique()))

# Identify which groups are C1/C2 vs C1/DETANO
c1_c2_groups = group_conditions[group_conditions == {"C1", "C2"}].index
c1_detano_groups = group_conditions[group_conditions == {"C1", "DETANO"}].index

# Build a boolean mask using the (date, field) MultiIndex
df_idx = df.set_index(["date", "field"]).index

df_c1_c2_chirp = df[df_idx.isin(c1_c2_groups)]
df_c1_detano_chirp = df[df_idx.isin(c1_detano_groups)]

print(f"C1/C2 groups: {len(c1_c2_groups)}")
print(f"C1/DETANO groups: {len(c1_detano_groups)}")

C1/C2 groups: 8
C1/DETANO groups: 9


In [6]:
dates = df["date"].unique()

for date in dates:

    df_date = df[df["date"] == date]
    fields = df_date["field"].unique()

    for field in fields:

        df_field = df_date[df_date["field"] == field]
        cond2 = df_field["cond2"].unique()

        df_field_filtered = df_field[df_field["qidx_gChirp"] > 0.35]

        print(f"{date}; {field}: Filtered: {len(df_field_filtered)}, All: {len(df_field)}, In %: {np.round(len(df_field_filtered)/len(df_field)*100, 0)}")

2025-09-08; XY1: Filtered: 45, All: 186, In %: 24.0
2025-09-08; XY2: Filtered: 62, All: 374, In %: 17.0
2025-09-08; XY3: Filtered: 137, All: 268, In %: 51.0
2025-09-08; XY4: Filtered: 78, All: 320, In %: 24.0
2025-09-08; XY6: Filtered: 36, All: 298, In %: 12.0
2025-09-08; XY7: Filtered: 431, All: 498, In %: 87.0
2025-09-08; XY8: Filtered: 384, All: 516, In %: 74.0
2025-10-23; XY7: Filtered: 266, All: 354, In %: 75.0
2025-10-23; XY8: Filtered: 36, All: 366, In %: 10.0
2025-10-23; XY1: Filtered: 83, All: 334, In %: 25.0
2025-10-23; XY2: Filtered: 159, All: 326, In %: 49.0
2025-10-23; XY3: Filtered: 104, All: 226, In %: 46.0
2025-10-23; XY4: Filtered: 378, All: 378, In %: 100.0
2025-10-23; XY5: Filtered: 326, All: 364, In %: 90.0
2025-10-23; XY6: Filtered: 224, All: 332, In %: 67.0
2025-11-19; XY4: Filtered: 348, All: 374, In %: 93.0
2025-11-19; XY5: Filtered: 135, All: 374, In %: 36.0


In [8]:
df_c1 = df[df["cond2"] == "C1"]

# Quality threshold for individual recordings
qidx_threshold = 0.35

# Minimum fraction of recordings that need to pass
quality_threshold = 0.30

# Calculate quality for each date-field pair
field_quality = (
    df_c1
    .assign(passes_quality=df_c1["qidx_gChirp"] > qidx_threshold)
    .groupby(["date", "field"])["passes_quality"]
    .mean()
)

# Keep only date-field pairs with >= 30% good recordings
good_fields = field_quality[field_quality >= quality_threshold].index

# Filter the original dataframe
df_c1_filtered = (
    df_c1.set_index(["date", "field"])
         .loc[good_fields]
         .reset_index()
)

print(f"Original rows: {len(df_c1)}")
print(f"Filtered rows: {len(df_c1_filtered)}")

Original rows: 2944
Filtered rows: 1892


In [12]:
keys = [
    "experimenter", "date", "exp_num", "raw_id",
    "field", "region", "cond1", "roi_id"
]

# 2. Retrieve the unique identifiers of the selected ROIs
selected_rois = df_c1_filtered[keys].drop_duplicates()

# 3. Retrieve all cond2 conditions for those same ROIs
df_all_conditions = df.merge(
    selected_rois,
    on=keys,
    how="inner"
)

In [13]:
# Determine the set of cond2 values per (date, field) group
group_conditions = df_all_conditions.groupby(["date", "field"])["cond2"].apply(lambda s: set(s.unique()))

# Identify which groups are C1/C2 vs C1/DETANO
c2_groups = group_conditions[group_conditions == {"C1", "C2"}].index
detano_groups = group_conditions[group_conditions == {"C1", "DETANO"}].index

# Build a boolean mask using the (date, field) MultiIndex
df_idx = df_all_conditions.set_index(["date", "field"]).index

df_c2 = df_all_conditions[df_idx.isin(c1_c2_groups)]
df_detano = df_all_conditions[df_idx.isin(c1_detano_groups)]

print(f"C1/C2 groups: {len(c2_groups)}")
print(f"C1/DETANO groups: {len(detano_groups)}")

C1/C2 groups: 5
C1/DETANO groups: 5


In [16]:
### DEFINE JSON STATE AND SAVE DATAFRAME ###

description = ["Average traces from Ca imaging data from IPL, GAD65 x Ai95GCaMP, field filtering (e.g., fields with ROIs gChirp > 0.35 > 0.3 were kicked out"]
inputs = {"source": "filtering_fields_gchirp.ipynb", "dataset": "../datasets/Processed/Averages_ClusterLabels_20261013.pkl","restriction": "No restriction used here"}
params = {"stimuli": "gChirp, lChirp, MB",
          "filtering": "quality restriction as described in the description"}

In [20]:
save_with_meta(df=df_all_conditions, path=f"../datasets/Processed/Averages_gChirpFieldsFiltered_{current_date}.pkl",
               description=description,
               inputs=inputs,
               params=params)